In [1]:
from __future__ import annotations

import os
from datetime import datetime
from getpass import getpass
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
cities = [
    {
        "city": "Warsaw",
        "country": "Poland",
        "latitude": 52.2297,
        "longitude": 21.0122,
    },
    {
        "city": "Krakow",
        "country": "Poland",
        "latitude": 50.0647,
        "longitude": 19.9450,
    },
    {
        "city": "Gdansk",
        "country": "Poland",
        "latitude": 54.3520,
        "longitude": 18.6466,
    },
    {
        "city": "Wroclaw",
        "country": "Poland",
        "latitude": 51.1079,
        "longitude": 17.0385,
    },
    {
        "city": "Poznan",
        "country": "Poland",
        "latitude": 52.4064,
        "longitude": 16.9252,
    },
]

In [5]:
def fetch_weather_forecast(
    *,
    latitude: float,
    longitude: float,
    forecast_days: int = 5,
    timezone: str = "Europe/Warsaw",
) -> dict[str, Any]:
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "apparent_temperature_max",
            "apparent_temperature_min",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max",
        ],
        "timezone": timezone,
        "forecast_days": forecast_days,
    }

    with httpx.Client(timeout=30.0) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [6]:
warsaw_forecast = fetch_weather_forecast(
    latitude=52.2297,
    longitude=21.0122,
    forecast_days=3,
)

warsaw_forecast.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

In [7]:
warsaw_forecast["daily"]

{'time': ['2026-06-03', '2026-06-04', '2026-06-05'],
 'weather_code': [3, 63, 55],
 'temperature_2m_max': [23.5, 18.7, 25.3],
 'temperature_2m_min': [13.8, 15.1, 12.9],
 'apparent_temperature_max': [22.8, 19.0, 25.2],
 'apparent_temperature_min': [12.5, 14.4, 11.2],
 'precipitation_sum': [0.0, 6.1, 3.88],
 'precipitation_probability_max': [2, 88, 80],
 'wind_speed_10m_max': [15.5, 15.1, 24.5]}

In [8]:
WEATHER_CODE_MAP = {
    0: "clear sky",
    1: "mainly clear",
    2: "partly cloudy",
    3: "overcast",
    45: "fog",
    48: "depositing rime fog",
    51: "light drizzle",
    53: "moderate drizzle",
    55: "dense drizzle",
    56: "light freezing drizzle",
    57: "dense freezing drizzle",
    61: "slight rain",
    63: "moderate rain",
    65: "heavy rain",
    66: "light freezing rain",
    67: "heavy freezing rain",
    71: "slight snow fall",
    73: "moderate snow fall",
    75: "heavy snow fall",
    77: "snow grains",
    80: "slight rain showers",
    81: "moderate rain showers",
    82: "violent rain showers",
    85: "slight snow showers",
    86: "heavy snow showers",
    95: "thunderstorm",
    96: "thunderstorm with slight hail",
    99: "thunderstorm with heavy hail",
}


def describe_weather_code(code: int) -> str:
    return WEATHER_CODE_MAP.get(code, f"unknown weather code {code}")

In [9]:
def normalize_forecast(
    *,
    city: str,
    country: str,
    latitude: float,
    longitude: float,
    api_response: dict[str, Any],
) -> list[dict[str, Any]]:
    daily = api_response["daily"]

    records = []

    for index, date_value in enumerate(daily["time"]):
        weather_code = daily["weather_code"][index]
        weather_description = describe_weather_code(weather_code)

        temp_max = daily["temperature_2m_max"][index]
        temp_min = daily["temperature_2m_min"][index]
        apparent_max = daily["apparent_temperature_max"][index]
        apparent_min = daily["apparent_temperature_min"][index]
        precipitation_sum = daily["precipitation_sum"][index]
        precipitation_probability = daily["precipitation_probability_max"][index]
        wind_speed_max = daily["wind_speed_10m_max"][index]

        summary = (
            f"Weather forecast for {city}, {country} on {date_value}: "
            f"{weather_description}. "
            f"Temperature from {temp_min}°C to {temp_max}°C. "
            f"Apparent temperature from {apparent_min}°C to {apparent_max}°C. "
            f"Precipitation sum {precipitation_sum} mm with maximum probability "
            f"{precipitation_probability}%. "
            f"Maximum wind speed {wind_speed_max} km/h."
        )

        if precipitation_probability >= 60 or precipitation_sum >= 3:
            travel_advice = "Take an umbrella or waterproof jacket."
        elif wind_speed_max >= 40:
            travel_advice = "Expect windy conditions and plan outdoor activities carefully."
        elif temp_max >= 28:
            travel_advice = "Stay hydrated and avoid long exposure to strong sun."
        elif temp_min <= 0:
            travel_advice = "Expect cold conditions and dress warmly."
        else:
            travel_advice = "Weather conditions look manageable for normal outdoor activity."

        records.append(
            {
                "city": city,
                "country": country,
                "forecast_date": datetime.fromisoformat(date_value),
                "weather_code": weather_code,
                "weather_description": weather_description,
                "temperature_max_c": float(temp_max),
                "temperature_min_c": float(temp_min),
                "apparent_temperature_max_c": float(apparent_max),
                "apparent_temperature_min_c": float(apparent_min),
                "precipitation_sum_mm": float(precipitation_sum),
                "precipitation_probability_max": int(precipitation_probability),
                "wind_speed_max_kmh": float(wind_speed_max),
                "latitude": float(latitude),
                "longitude": float(longitude),
                "summary": summary,
                "travel_advice": travel_advice,
            }
        )

    return records

In [10]:
forecast_records = []

for city_config in cities:
    api_response = fetch_weather_forecast(
        latitude=city_config["latitude"],
        longitude=city_config["longitude"],
        forecast_days=5,
    )

    city_records = normalize_forecast(
        city=city_config["city"],
        country=city_config["country"],
        latitude=city_config["latitude"],
        longitude=city_config["longitude"],
        api_response=api_response,
    )

    forecast_records.extend(city_records)

print("Total records:", len(forecast_records))

Total records: 25


In [11]:
for record in forecast_records[:5]:
    print(record)
    print("-" * 80)

{'city': 'Warsaw', 'country': 'Poland', 'forecast_date': datetime.datetime(2026, 6, 3, 0, 0), 'weather_code': 3, 'weather_description': 'overcast', 'temperature_max_c': 23.5, 'temperature_min_c': 13.8, 'apparent_temperature_max_c': 22.8, 'apparent_temperature_min_c': 12.5, 'precipitation_sum_mm': 0.0, 'precipitation_probability_max': 2, 'wind_speed_max_kmh': 15.5, 'latitude': 52.2297, 'longitude': 21.0122, 'summary': 'Weather forecast for Warsaw, Poland on 2026-06-03: overcast. Temperature from 13.8°C to 23.5°C. Apparent temperature from 12.5°C to 22.8°C. Precipitation sum 0.0 mm with maximum probability 2%. Maximum wind speed 15.5 km/h.', 'travel_advice': 'Weather conditions look manageable for normal outdoor activity.'}
--------------------------------------------------------------------------------
{'city': 'Warsaw', 'country': 'Poland', 'forecast_date': datetime.datetime(2026, 6, 4, 0, 0), 'weather_code': 63, 'weather_description': 'moderate rain', 'temperature_max_c': 18.7, 'tempe

In [12]:
COLLECTION_NAME = "WeatherForecast"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

forecasts = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="city",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="country",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="forecast_date",
            data_type=wvc.config.DataType.DATE,
        ),
        wvc.config.Property(
            name="weather_code",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="weather_description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="temperature_max_c",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="temperature_min_c",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="apparent_temperature_max_c",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="apparent_temperature_min_c",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="precipitation_sum_mm",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="precipitation_probability_max",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="wind_speed_max_kmh",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="latitude",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="longitude",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="summary",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="travel_advice",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: WeatherForecast


In [13]:
result = forecasts.data.insert_many(forecast_records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/weaviate/warnings.py:256: UserWarning: Con002: You are using the datetime object 2026-06-03 00:00:00 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warnings.warn(
/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/weaviate/warnings.py:256: UserWarning: Con002: You are using the datetime object 2026-06-04 00:00:00 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warning

Import completed.


In [14]:
response = forecasts.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 25


In [15]:
response = forecasts.query.fetch_objects(
    limit=10,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "temperature_min_c",
        "temperature_max_c",
        "precipitation_probability_max",
        "wind_speed_max_kmh",
        "summary",
        "travel_advice",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("City:", obj.properties["city"])
    print("Date:", obj.properties["forecast_date"])
    print("Weather:", obj.properties["weather_description"])
    print("Temp:", obj.properties["temperature_min_c"], "-", obj.properties["temperature_max_c"])
    print("Rain probability:", obj.properties["precipitation_probability_max"])
    print("Wind:", obj.properties["wind_speed_max_kmh"])
    print("Advice:", obj.properties["travel_advice"])
    print("-" * 80)

UUID: 029d9b6c-c0e1-4159-a188-2a4e8aba9a08
City: Warsaw
Date: 2026-06-06 00:00:00+00:00
Weather: light drizzle
Temp: 10.7 - 22.1
Rain probability: 41
Wind: 12.9
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
UUID: 055c7329-9f6a-413d-961a-c3f7f7620b86
City: Gdansk
Date: 2026-06-07 00:00:00+00:00
Weather: slight rain showers
Temp: 12.6 - 22.3
Rain probability: 38
Wind: 10.3
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
UUID: 1a26a5cd-3b57-4540-b3f4-b2d42a52bd9d
City: Krakow
Date: 2026-06-05 00:00:00+00:00
Weather: light drizzle
Temp: 12.5 - 26.6
Rain probability: 90
Wind: 21.2
Advice: Take an umbrella or waterproof jacket.
--------------------------------------------------------------------------------
UUID: 1b4a2bcc-b3a2-402c-8b8a-1fb9ad96810d
City: Gdansk
Date: 2026-06-06 00:0

In [16]:
response = forecasts.query.near_text(
    query="good weather for walking outside and sightseeing",
    limit=5,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "temperature_max_c",
        "precipitation_probability_max",
        "wind_speed_max_kmh",
        "summary",
        "travel_advice",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("City:", obj.properties["city"])
    print("Date:", obj.properties["forecast_date"])
    print("Weather:", obj.properties["weather_description"])
    print("Temperature max:", obj.properties["temperature_max_c"])
    print("Rain probability:", obj.properties["precipitation_probability_max"])
    print("Wind:", obj.properties["wind_speed_max_kmh"])
    print("Advice:", obj.properties["travel_advice"])
    print("-" * 80)

Distance: 0.6664959788322449
City: Warsaw
Date: 2026-06-03 00:00:00+00:00
Weather: overcast
Temperature max: 23.5
Rain probability: 2
Wind: 15.5
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6708749532699585
City: Poznan
Date: 2026-06-04 00:00:00+00:00
Weather: slight rain
Temperature max: 25.7
Rain probability: 23
Wind: 13.3
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6754502058029175
City: Warsaw
Date: 2026-06-06 00:00:00+00:00
Weather: light drizzle
Temperature max: 22.1
Rain probability: 41
Wind: 12.9
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6774562001228333
City: Poznan
Date: 2026-06-06 00:00:00+00:00
Weather: overcast
Temp

In [17]:
response = forecasts.query.fetch_objects(
    filters=Filter.by_property("city").equal("Warsaw"),
    limit=10,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "temperature_min_c",
        "temperature_max_c",
        "precipitation_probability_max",
        "summary",
    ],
)

for obj in response.objects:
    print("City:", obj.properties["city"])
    print("Date:", obj.properties["forecast_date"])
    print("Weather:", obj.properties["weather_description"])
    print("Temperature:", obj.properties["temperature_min_c"], "-", obj.properties["temperature_max_c"])
    print("Rain probability:", obj.properties["precipitation_probability_max"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

City: Warsaw
Date: 2026-06-04 00:00:00+00:00
Weather: moderate rain
Temperature: 15.1 - 18.7
Rain probability: 88
Summary: Weather forecast for Warsaw, Poland on 2026-06-04: moderate rain. Temperature from 15.1°C to 18.7°C. Apparent temperature from 14.4°C to 19.0°C. Precipitation sum 6.1 mm with maximum probability 88%. Maximum wind speed 15.1 km/h.
--------------------------------------------------------------------------------
City: Warsaw
Date: 2026-06-03 00:00:00+00:00
Weather: overcast
Temperature: 13.8 - 23.5
Rain probability: 2
Summary: Weather forecast for Warsaw, Poland on 2026-06-03: overcast. Temperature from 13.8°C to 23.5°C. Apparent temperature from 12.5°C to 22.8°C. Precipitation sum 0.0 mm with maximum probability 2%. Maximum wind speed 15.5 km/h.
--------------------------------------------------------------------------------
City: Warsaw
Date: 2026-06-05 00:00:00+00:00
Weather: dense drizzle
Temperature: 12.9 - 25.3
Rain probability: 80
Summary: Weather forecast for 

In [18]:
response = forecasts.query.fetch_objects(
    filters=Filter.by_property("precipitation_probability_max").greater_or_equal(60),
    limit=20,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "precipitation_sum_mm",
        "precipitation_probability_max",
        "travel_advice",
    ],
)

for obj in response.objects:
    print("City:", obj.properties["city"])
    print("Date:", obj.properties["forecast_date"])
    print("Weather:", obj.properties["weather_description"])
    print("Rain mm:", obj.properties["precipitation_sum_mm"])
    print("Rain probability:", obj.properties["precipitation_probability_max"])
    print("Advice:", obj.properties["travel_advice"])
    print("-" * 80)

City: Warsaw
Date: 2026-06-04 00:00:00+00:00
Weather: moderate rain
Rain mm: 6.1
Rain probability: 88
Advice: Take an umbrella or waterproof jacket.
--------------------------------------------------------------------------------
City: Warsaw
Date: 2026-06-05 00:00:00+00:00
Weather: dense drizzle
Rain mm: 3.88
Rain probability: 80
Advice: Take an umbrella or waterproof jacket.
--------------------------------------------------------------------------------
City: Krakow
Date: 2026-06-03 00:00:00+00:00
Weather: slight rain
Rain mm: 4.7
Rain probability: 89
Advice: Take an umbrella or waterproof jacket.
--------------------------------------------------------------------------------
City: Krakow
Date: 2026-06-04 00:00:00+00:00
Weather: slight rain
Rain mm: 8.2
Rain probability: 86
Advice: Take an umbrella or waterproof jacket.
--------------------------------------------------------------------------------
City: Krakow
Date: 2026-06-05 00:00:00+00:00
Weather: light drizzle
Rain mm: 0.2
Ra

In [19]:
filters = (
    Filter.by_property("precipitation_probability_max").less_or_equal(40)
    & Filter.by_property("wind_speed_max_kmh").less_or_equal(35)
    & Filter.by_property("temperature_max_c").greater_or_equal(15)
)

response = forecasts.query.near_text(
    query="pleasant weather for outdoor sightseeing and walking",
    filters=filters,
    limit=10,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "temperature_max_c",
        "precipitation_probability_max",
        "wind_speed_max_kmh",
        "summary",
        "travel_advice",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("City:", obj.properties["city"])
    print("Date:", obj.properties["forecast_date"])
    print("Weather:", obj.properties["weather_description"])
    print("Max temp:", obj.properties["temperature_max_c"])
    print("Rain probability:", obj.properties["precipitation_probability_max"])
    print("Wind:", obj.properties["wind_speed_max_kmh"])
    print("Advice:", obj.properties["travel_advice"])
    print("-" * 80)

Distance: 0.6375063061714172
City: Poznan
Date: 2026-06-04 00:00:00+00:00
Weather: slight rain
Max temp: 25.7
Rain probability: 23
Wind: 13.3
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6411285996437073
City: Warsaw
Date: 2026-06-03 00:00:00+00:00
Weather: overcast
Max temp: 23.5
Rain probability: 2
Wind: 15.5
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6464214324951172
City: Poznan
Date: 2026-06-06 00:00:00+00:00
Weather: overcast
Max temp: 23.3
Rain probability: 24
Wind: 7.6
Advice: Weather conditions look manageable for normal outdoor activity.
--------------------------------------------------------------------------------
Distance: 0.6491778492927551
City: Poznan
Date: 2026-06-07 00:00:00+00:00
Weather: slight rain showers
Max temp: 25.0
Rain 

In [20]:
filters = (
    Filter.by_property("precipitation_probability_max").less_or_equal(50)
    & Filter.by_property("wind_speed_max_kmh").less_or_equal(40)
)

response = forecasts.generate.near_text(
    query="best city and day for walking outside, sightseeing or a short trip",
    filters=filters,
    grouped_task=(
        "Recommend the best city and forecast day for outdoor walking or sightseeing. "
        "Use only the retrieved weather forecast records. "
        "Mention city, date, weather, temperature, rain probability, wind and practical advice. "
        "Keep the answer concise."
    ),
    limit=10,
    return_properties=[
        "city",
        "forecast_date",
        "weather_description",
        "temperature_min_c",
        "temperature_max_c",
        "precipitation_probability_max",
        "wind_speed_max_kmh",
        "summary",
        "travel_advice",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["city"],
        "|",
        obj.properties["forecast_date"],
        "|",
        obj.properties["weather_description"],
        "| rain:",
        obj.properties["precipitation_probability_max"],
        "%",
    )

**Best City for Outdoor Walking: Wroclaw, Poland**  
**Date:** June 7, 2026  
**Weather:** Overcast  
**Temperature:** High of 25.9°C, Low of 15.4°C  
**Rain Probability:** 23% (0 mm expected)  
**Wind:** Max speed 13.1 km/h  

**Practical Advice:** Conditions are favorable for outdoor activities; bring water and wear comfortable walking shoes. Enjoy your time exploring Wroclaw!

Sources:
- Poznan | 2026-06-04 00:00:00+00:00 | slight rain | rain: 23 %
- Poznan | 2026-06-07 00:00:00+00:00 | slight rain showers | rain: 24 %
- Warsaw | 2026-06-03 00:00:00+00:00 | overcast | rain: 2 %
- Poznan | 2026-06-06 00:00:00+00:00 | overcast | rain: 24 %
- Krakow | 2026-06-07 00:00:00+00:00 | light drizzle | rain: 26 %
- Warsaw | 2026-06-06 00:00:00+00:00 | light drizzle | rain: 41 %
- Wroclaw | 2026-06-04 00:00:00+00:00 | overcast | rain: 3 %
- Warsaw | 2026-06-07 00:00:00+00:00 | moderate drizzle | rain: 33 %
- Krakow | 2026-06-06 00:00:00+00:00 | overcast | rain: 14 %
- Wroclaw | 2026-06-07 00:00

In [21]:
def recommend_weather(
    user_need: str,
    *,
    city: str | None = None,
    max_rain_probability: int | None = None,
    max_wind_speed: float | None = None,
    min_temperature: float | None = None,
    limit: int = 10,
) -> None:
    filters = None

    if city is not None:
        filters = Filter.by_property("city").equal(city)

    if max_rain_probability is not None:
        rain_filter = Filter.by_property("precipitation_probability_max").less_or_equal(
            max_rain_probability
        )
        filters = rain_filter if filters is None else filters & rain_filter

    if max_wind_speed is not None:
        wind_filter = Filter.by_property("wind_speed_max_kmh").less_or_equal(max_wind_speed)
        filters = wind_filter if filters is None else filters & wind_filter

    if min_temperature is not None:
        temp_filter = Filter.by_property("temperature_max_c").greater_or_equal(min_temperature)
        filters = temp_filter if filters is None else filters & temp_filter

    response = forecasts.generate.near_text(
        query=user_need,
        filters=filters,
        grouped_task=(
            "Act as a practical weather assistant. "
            "Use only the retrieved forecast records. "
            "Recommend the best matching day and city. "
            "Mention date, weather description, temperature, rain probability, wind and advice. "
            "If the retrieved records are not suitable, say so clearly."
        ),
        limit=limit,
        return_properties=[
            "city",
            "country",
            "forecast_date",
            "weather_description",
            "temperature_min_c",
            "temperature_max_c",
            "precipitation_probability_max",
            "precipitation_sum_mm",
            "wind_speed_max_kmh",
            "summary",
            "travel_advice",
        ],
    )

    print("USER NEED:", user_need)
    print("CITY:", city)
    print("MAX RAIN:", max_rain_probability)
    print("MAX WIND:", max_wind_speed)
    print("MIN TEMPERATURE:", min_temperature)
    print("-" * 80)

    print(response.generated)

    print("\nRetrieved records:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["city"],
            "|",
            obj.properties["forecast_date"],
            "|",
            obj.properties["weather_description"],
            "| temp max:",
            obj.properties["temperature_max_c"],
            "| rain:",
            obj.properties["precipitation_probability_max"],
            "| wind:",
            obj.properties["wind_speed_max_kmh"],
        )

In [22]:
recommend_weather(
    "I want to go for a long walk and avoid rain",
    city="Warsaw",
    max_rain_probability=40,
    max_wind_speed=35,
    min_temperature=10,
)

USER NEED: I want to go for a long walk and avoid rain
CITY: Warsaw
MAX RAIN: 40
MAX WIND: 35
MIN TEMPERATURE: 10
--------------------------------------------------------------------------------
Based on the retrieved forecast records, I recommend the following day and city:

**City:** Warsaw, Poland  
**Date:** June 3, 2026  
**Weather Description:** Overcast  
**Temperature:** Max 23.5°C, Min 13.8°C  
**Rain Probability:** 2% (No precipitation expected)  
**Wind:** Max 15.5 km/h  
**Travel Advice:** Weather conditions look manageable for normal outdoor activity.

This day offers pleasant temperatures without the concern of rain, making it suitable for outdoor plans.

Retrieved records:
- Warsaw | 2026-06-07 00:00:00+00:00 | moderate drizzle | temp max: 23.2 | rain: 33 | wind: 9.2
- Warsaw | 2026-06-03 00:00:00+00:00 | overcast | temp max: 23.5 | rain: 2 | wind: 15.5


In [23]:
recommend_weather(
    "I want the best city for sightseeing and outdoor activity",
    max_rain_probability=50,
    max_wind_speed=40,
    min_temperature=12,
)

USER NEED: I want the best city for sightseeing and outdoor activity
CITY: None
MAX RAIN: 50
MAX WIND: 40
MIN TEMPERATURE: 12
--------------------------------------------------------------------------------
Based on the retrieved forecast records, the best matching day and city for manageable weather conditions is:

**City:** Krakow, Poland  
**Date:** June 6, 2026  
**Weather Description:** Overcast  
**Temperature:** Max 23.9°C, Min 10.4°C  
**Rain Probability:** 14%  
**Wind:** Max Speed 8.9 km/h  
**Advice:** Weather conditions look manageable for normal outdoor activity. 

This day provides pleasant temperatures with minimal risk of rain, making it suitable for outdoor plans.

Retrieved records:
- Krakow | 2026-06-06 00:00:00+00:00 | overcast | temp max: 23.9 | rain: 14 | wind: 8.9
- Krakow | 2026-06-07 00:00:00+00:00 | light drizzle | temp max: 25.2 | rain: 26 | wind: 14.7
- Poznan | 2026-06-06 00:00:00+00:00 | overcast | temp max: 23.3 | rain: 24 | wind: 7.6
- Poznan | 2026-06-0

In [24]:
recommend_weather(
    "I want weather suitable for running outside",
    max_rain_probability=40,
    max_wind_speed=30,
    min_temperature=8,
)

USER NEED: I want weather suitable for running outside
CITY: None
MAX RAIN: 40
MAX WIND: 30
MIN TEMPERATURE: 8
--------------------------------------------------------------------------------
Based on the retrieved forecast records, here’s the best matching day and city for outdoor activities:

### City: Wroclaw, Poland
- **Date:** June 7, 2026
- **Weather Description:** Overcast
- **Temperature:** 25.9°C (Max) / 15.4°C (Min)
- **Rain Probability:** 23% (Precipitation sum: 0.0 mm)
- **Wind:** Max wind speed 13.1 km/h
- **Advice:** Weather conditions look manageable for normal outdoor activity.

This day in Wroclaw offers good temperatures and low chances of rain, making it suitable for outdoor activities.

Retrieved records:
- Poznan | 2026-06-04 00:00:00+00:00 | slight rain | temp max: 25.7 | rain: 23 | wind: 13.3
- Poznan | 2026-06-07 00:00:00+00:00 | slight rain showers | temp max: 25.0 | rain: 24 | wind: 11.6
- Poznan | 2026-06-06 00:00:00+00:00 | overcast | temp max: 23.3 | rain: 

In [25]:
response = forecasts.aggregate.over_all(total_count=True)

print("Total forecast records:", response.total_count)

Total forecast records: 25


In [26]:
client.close()